In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load csv results
hrv_results_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv')
pupil_results_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv')
duration_results_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# combine with suffixes
combined_results = pd.concat([
    hrv_results_df.set_index('Participant').add_suffix('_HRV_SDNN'),
    pupil_results_df.set_index('Participant').add_suffix('_Pupil_STD'),
    duration_results_df.set_index('Participant').add_suffix('_Duration_STD'),
], axis=1)

# correlation matrix
correlation_matrix = combined_results.corr(method='spearman')

# plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of HRV SDNN, Pupil Dilation STD, and Duration STD')
plt.show()
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load hrv + pupil
hrv_results_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv')
pupil_results_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv')

# indexed lookup
hrv_idx = hrv_results_df.set_index('Participant')
pupil_idx = pupil_results_df.set_index('Participant')

for pid in hrv_idx.index:
    for s in [1, 2, 3]:
        col = f'Session {s:02d}'
        print(f"Participant {pid}, Session {s}, HRV SDNN: {hrv_idx.loc[pid, col]:.2f}")
        print(f"Participant {pid}, Session {s}, Pupil Dilation STD: {pupil_idx.loc[pid, col]:.2f}")

# combine with suffixes
combined_results = pd.concat([
    hrv_results_df.set_index('Participant').add_suffix('_HRV_SDNN'),
    pupil_results_df.set_index('Participant').add_suffix('_Pupil_STD'),
], axis=1)

# correlation matrix
correlation_matrix = combined_results.corr(method='spearman')

# plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of HRV SDNN and Pupil Dilation STD')
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load data
hrv_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv').set_index('Participant')
pupil_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv').set_index('Participant')
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv').set_index('Participant')

# combine all
combined = pd.concat([
    hrv_df.add_suffix('_HRV_SDNN'),
    pupil_df.add_suffix('_Pupil_STD'),
    duration_df.add_suffix('_Duration_STD'),
], axis=1)

cols = combined.columns
n = len(cols)
n_obs = len(combined)

# pairwise correlations + raw p-values
r_matrix = np.zeros((n, n))
p_matrix = np.ones((n, n))
for i in range(n):
    for j in range(n):
        r, p = stats.pearsonr(combined.iloc[:, i], combined.iloc[:, j])
        r_matrix[i, j] = r
        p_matrix[i, j] = p

r_df = pd.DataFrame(r_matrix, index=cols, columns=cols)

# Benjamini-Hochberg FDR across the unique off-diagonal pairs only.
# With n_obs = 10 participants and dozens of pairwise tests, uncorrected
# stars overstate significance; we control the false-discovery rate instead.
iu = np.triu_indices(n, k=1)
raw_p = p_matrix[iu]
order = np.argsort(raw_p)
ranks = np.empty_like(order)
ranks[order] = np.arange(1, len(raw_p) + 1)
q = raw_p * len(raw_p) / ranks
# enforce monotonicity of BH q-values
q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
q_adj = np.empty_like(q)
q_adj[order] = np.clip(q_sorted, 0, 1)

q_matrix = np.ones((n, n))
q_matrix[iu] = q_adj
q_matrix[(iu[1], iu[0])] = q_adj

def star(qv):
    return "***" if qv < 0.001 else "**" if qv < 0.01 else "*" if qv < 0.05 else ""

annot = np.empty((n, n), dtype=object)
for i in range(n):
    for j in range(n):
        s = "" if i == j else star(q_matrix[i, j])
        annot[i, j] = f"{r_df.iloc[i, j]:.2f}{s}"

plt.figure(figsize=(14, 11))
sns.heatmap(r_df, annot=annot, fmt='', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title(f'Correlation Matrix (Pearson r, n={n_obs}); '
          f'stars = Benjamini-Hochberg FDR q-value (* q<.05, ** q<.01, *** q<.001)')
plt.tight_layout()
plt.show()
plt.close()
